# Rocket Lab - Atividade 1
<br>

## Silver para Gold

Terceiro notebook da pipeline da atividade, com os seguintes objetivos:
- ler as tabelas tratadas da camada Silver;
- consolidar os dados em Data Marts analíticos da camada Gold;
- calcular métricas de negócio para visão comercial e satisfação de clientes;
- exibir recortes analíticos em tela com display().

Nesta etapa, os dados da camada Silver são organizados em agregações finais para consumo analítico e acompanhamento de indicadores de negócio.
___

## Etapa de Imports, definições que o notebook vai usar e criação e configurações da camada Gold
___

In [0]:
# Célula somente para imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Criação schema da camada Gold
spark.sql("CREATE SCHEMA IF NOT EXISTS ecommerce_olist.gold")

spark.sql("SHOW CATALOGS").show()
spark.sql("SHOW SCHEMAS IN ecommerce_olist").show()

+---------------+
|        catalog|
+---------------+
|ecommerce_olist|
|        samples|
|         system|
|      workspace|
+---------------+

+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|              gold|
|information_schema|
|           landing|
|            silver|
+------------------+



In [0]:
# Célula apenas para confirmar a utilização do catálogo e schema certos
spark.sql("USE CATALOG ecommerce_olist")
spark.sql("USE SCHEMA gold")

DataFrame[]

In [0]:
# Definições dos caminhos e leitura das tabelas silver para dfs
volume_path = "/Volumes/ecommerce_olist/landing/dados_olist"

catalogo = "ecommerce_olist"
schema = "gold"

df_fat_pedido_total = spark.table("silver.fat_pedido_total")
df_fat_itens_pedidos = spark.table("silver.fat_itens_pedidos")
df_fat_avaliacoes_pedidos = spark.table("silver.fat_avaliacoes_pedidos")
df_dim_produtos = spark.table("silver.dim_produtos")
df_dim_vendedores = spark.table("silver.dim_vendedores")

## Etapa de consolidação e estruturação dos Data Marts da camada Gold + Resultados
___

### Visão Comercial e Volume de Produtos


#### fat_vendas_comercial

In [0]:
# Relaciona cada item do pedido com sua categoria de produto
df_itens_com_categoria = (
    df_fat_itens_pedidos.alias("it")
    .join(
        df_dim_produtos.select("id_produto", "categoria_produto").alias("pr"),
        on="id_produto",
        how="left"
    ) # Faz o join com a tabela de dimensão de produtos para conseguir pegar a categoria do produto
    .withColumn(
        "valor_item_total_brl",
        F.round(F.col("preco_BRL") + F.col("preco_frete"), 2)
    ) # Faz a soma do custo total para o cliente, somando o preço do produto com o frete
)

In [0]:
# Calcula o valor total de itens por pedido e categoria
df_categoria_por_pedido = (
    df_itens_com_categoria
    .groupBy("id_pedido", "categoria_produto") # Agrupa os dados por pedido e categoria para resumir os itens dentro de cada combinação
    .agg(
        F.count("*").alias("qtd_itens_vendidos"), # Conta quantos itens daquela categoria existem no pedido
        F.round(F.sum("valor_item_total_brl"), 2).alias("valor_categoria_itens_brl") # Soma o valor total de todos os itens daquela categoria
    )
)

In [0]:
# Calcula o valor total de itens de cada pedido
df_total_itens_pedido = (
    df_itens_com_categoria
    .groupBy("id_pedido") # Agrupa os dados por pedido para consolidar o valor total de todos os itens daquele pedido
    .agg(
        F.round(F.sum("valor_item_total_brl"), 2).alias("valor_total_itens_pedido_brl") 
    )
)

In [0]:
# Junta as informações de categoria com os dados consolidados dos pedidos
df_base_vendas_comercial = (
    df_categoria_por_pedido.alias("cat")
    .join(
        df_total_itens_pedido.alias("tot"),
        on="id_pedido",
        how="inner"
    ) # Junta com o total de itens do pedido para ter, na mesma base, o valor da categoria e o valor total do pedido em itens
    .join(
        df_fat_pedido_total.select(
            "id_pedido",
            "data_pedido",
            "valor_total_pago_brl",
            "valor_total_pago_usd"
        ).alias("ped"),
        on="id_pedido",
        how="inner"
    ) # Junta com a tabela consolidada de pedidos para trazer a data da compra e os valores totais pagos em BRL e USD
    .withColumn("ano_venda", F.year("data_pedido")) 
    .withColumn("mes_venda", F.month("data_pedido")) 
)

In [0]:
# Calcula a parcela da receita de cada categoria dentro do pedido
df_base_vendas_comercial = (
    df_base_vendas_comercial
    .withColumn(
        "receita_rateada_brl",
        F.when(
            F.col("valor_total_itens_pedido_brl") > 0,
            F.col("valor_total_pago_brl") * (
                F.col("valor_categoria_itens_brl") / F.col("valor_total_itens_pedido_brl")
            )
        ).otherwise(F.lit(0))
    ) # Calcula quanto da receita total em BRL corresponde à categoria, com base na participação dela no valor total dos itens do pedido
    .withColumn(
        "receita_rateada_usd",
        F.when(
            F.col("valor_total_itens_pedido_brl") > 0,
            F.col("valor_total_pago_usd") * (
                F.col("valor_categoria_itens_brl") / F.col("valor_total_itens_pedido_brl")
            )
        ).otherwise(F.lit(0))
    ) # Calcula quanto da receita total em USD corresponde à categoria, seguindo a mesma proporção usada em BRL
)

In [0]:
# Criação da tabela fato comercial agrupando os dados por ano, mês e categoria de produto
df_fat_vendas_comercial = (
    df_base_vendas_comercial
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    .agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.sum("qtd_itens_vendidos").alias("qtd_itens_vendidos"),
        F.round(F.sum("receita_rateada_brl"), 2).alias("receita_total_brl"),
        F.round(F.sum("receita_rateada_usd"), 2).alias("receita_total_usd")
    )
    .withColumn(
        "ticket_medio_brl",
        F.round(
            F.col("receita_total_brl") / F.col("total_pedidos"),
            2
        )
    )
    .orderBy("ano_venda", "mes_venda", "categoria_produto")
)

In [0]:
# Apenas visualização do df
display(df_fat_vendas_comercial.orderBy("ano_venda", "mes_venda", "categoria_produto").limit(10))
df_fat_vendas_comercial.printSchema()

In [0]:
# Salvando a tabela
df_fat_vendas_comercial.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{schema}.fat_vendas_comercial")

#### Rankings Comerciais

In [0]:
# Junta os itens dos pedidos com as informações dos produtos
df_base_ranking_produtos = (
    df_fat_itens_pedidos.alias("it")
    .join(
        df_dim_produtos.select("id_produto", "nome_produto", "categoria_produto").alias("pr"),
        on="id_produto",
        how="left"
    ) # Faz o join com a dimensão de produtos para trazer o nome e a categoria de cada produto
)

In [0]:
# Calcula a quantidade total vendida de cada produto
df_ranking_produtos = (
    df_base_ranking_produtos
    .groupBy("id_produto", "nome_produto", "categoria_produto") # Agrupa os dados no nível do produto
    .agg(
        F.count("*").alias("quantidade_vendida")
    ) # Conta quantas vezes o produto apareceu nos itens dos pedidos
)

##### Resultados

In [0]:
# Seleciona os 5 produtos mais vendidos
df_top_5_produtos_mais_vendidos = (
    df_ranking_produtos
    .orderBy(
        F.col("quantidade_vendida").desc(),
    ) # Ordena da maior para a menor quantidade vendida
    .select("nome_produto", "categoria_produto", "quantidade_vendida")
    .limit(5)
)

display(df_top_5_produtos_mais_vendidos)

nome_produto,categoria_produto,quantidade_vendida
Estante de Livros Luxo,moveis_decoracao,527
Cobertor Cinza,cama_mesa_banho,488
Cortador de Grama Branco,ferramentas_jardim,484
Kit de Ferramentas Ultra,ferramentas_jardim,392
Kit de Ferramentas Master,ferramentas_jardim,388


In [0]:
# Seleciona os 5 produtos menos vendidos
df_top_5_produtos_menos_vendidos = (
    df_ranking_produtos
    .orderBy(
        F.col("quantidade_vendida").asc(),
    ) # Ordena da menor para a maior quantidade vendida
    .select("nome_produto", "categoria_produto", "quantidade_vendida")
    .limit(5)
)

display(df_top_5_produtos_menos_vendidos)

nome_produto,categoria_produto,quantidade_vendida
Cortador de Grama,ferramentas_jardim,1
Secador de Cabelo Verde,beleza_saude,1
Cadeira de Escritório Básico,moveis_decoracao,1
Item Básico Ultra,SEM CATEGORIA,1
Cadeira para Auto Max,bebes,1


### Satisfação de Clientes e Qualidade de Parceiros

#### fat_avaliacoes_clientes

In [0]:
# Relaciona cada item do pedido com a categoria do produto e as informações do vendedor
df_base_itens_avaliacoes = (
    df_fat_itens_pedidos.alias("it")
    .join(
        df_dim_produtos.select("id_produto", "categoria_produto").alias("pr"),
        on="id_produto",
        how="left"
    ) # Faz o join com a dimensão de produtos para trazer a categoria do produto
    .join(
        df_dim_vendedores.select("id_vendedor", "nome_vendedor", "estado").alias("ve"),
        on="id_vendedor",
        how="left"
    )
)

In [0]:
# Mantém apenas combinações únicas de pedido, categoria e vendedor
df_base_unica_avaliacoes = (
    df_base_itens_avaliacoes
    .select("id_pedido", "categoria_produto", "nome_vendedor", "estado")
    .dropDuplicates()
) 

In [0]:
# Junta a base única de pedidos com a tabela de avaliações
df_base_avaliacoes_clientes = (
    df_base_unica_avaliacoes.alias("base")
    .join(
        df_fat_avaliacoes_pedidos.select("id_pedido", "nota_avaliacao").alias("av"),
        on="id_pedido",
        how="inner" # Como é uma análise de avaliações, optei por registrar somente os pedidos que possuem avaliação
    ) # Junta com a tabela de avaliações para trazer a nota da avaliação
)

In [0]:
# Cria colunas auxiliares para facilitar o cálculo das métricas de satisfação
df_base_avaliacoes_clientes = (
    df_base_avaliacoes_clientes
    .withColumn(
        "avaliacao_positiva",
        F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)
    ) # Marca como positiva as avaliações com nota maior ou igual a 4
    .withColumn(
        "avaliacao_negativa",
        F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)
    ) # Marca como negativa as avaliações com nota menor ou igual a 2
)

In [0]:
# Criação da tabela fato de avaliações da camada Gold
df_fat_avaliacoes_clientes = (
    df_base_avaliacoes_clientes
    .groupBy("categoria_produto", "nome_vendedor", "estado") # Agrupa os dados pela granularidade pedida no requisito
    .agg(
        F.count("*").alias("total_avaliacoes"), # Conta o total de avaliações em cada grupo
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"), # Calcula a média das notas de avaliação
        F.sum("avaliacao_positiva").alias("total_avaliacoes_positivas"), # Soma a quantidade de avaliações positivas
        F.sum("avaliacao_negativa").alias("total_avaliacoes_negativas") # Soma a quantidade de avaliações negativas
    )
    .withColumn(
        "percentual_satisfacao",
        F.round(
            (F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes")) * 100,
            2
        )
    ) # Calcula o percentual de satisfação com base na proporção de avaliações positivas
    .orderBy("categoria_produto", "nome_vendedor", "estado")
)

In [0]:
display(df_fat_avaliacoes_clientes.limit(10))
df_fat_avaliacoes_clientes.printSchema()

categoria_produto,nome_vendedor,estado,total_avaliacoes,avaliacao_media,total_avaliacoes_positivas,total_avaliacoes_negativas,percentual_satisfacao
SEM CATEGORIA,Agatha Moura,PR,3,4.67,3,0,100.0
SEM CATEGORIA,Alana Araújo,SP,1,1.0,0,1,0.0
SEM CATEGORIA,Alexandre Aparecida,SP,2,4.5,2,0,100.0
SEM CATEGORIA,Alexia Garcia,SP,1,3.0,0,0,0.0
SEM CATEGORIA,Alexia Nascimento,RS,2,5.0,2,0,100.0
SEM CATEGORIA,Alexia Rodrigues,SC,2,1.0,0,2,0.0
SEM CATEGORIA,Alice Aparecida,RS,2,4.5,2,0,100.0
SEM CATEGORIA,Alice da Cunha,SP,9,4.67,9,0,100.0
SEM CATEGORIA,Alícia Carvalho,SP,7,3.86,6,1,85.71
SEM CATEGORIA,Alícia Marques,PR,1,2.0,0,1,0.0


root
 |-- categoria_produto: string (nullable = true)
 |-- nome_vendedor: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- total_avaliacoes: long (nullable = false)
 |-- avaliacao_media: double (nullable = true)
 |-- total_avaliacoes_positivas: long (nullable = true)
 |-- total_avaliacoes_negativas: long (nullable = true)
 |-- percentual_satisfacao: double (nullable = true)



In [0]:
df_fat_avaliacoes_clientes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{schema}.fat_avaliacoes_clientes")

#### Rankings de Qualidade

In [0]:
# Relaciona os itens dos pedidos com produto, categoria, vendedor e nota da avaliação
df_base_rankings_qualidade = (
    df_fat_itens_pedidos.alias("it")
    .join(
        df_dim_produtos.select("id_produto", "nome_produto", "categoria_produto").alias("pr"),
        on="id_produto",
        how="left"
    ) # Faz o join com a dimensão de produtos para trazer nome e categoria do produto
    .join(
        df_dim_vendedores.select("id_vendedor", "nome_vendedor", "estado").alias("ve"),
        on="id_vendedor",
        how="left"
    ) # Faz o join com a dimensão de vendedores para trazer nome e estado do vendedor
    .join(
        df_fat_avaliacoes_pedidos.select("id_pedido", "nota_avaliacao").alias("av"),
        on="id_pedido",
        how="inner"
    ) # Junta com a tabela de avaliações para trazer a nota dada ao pedido
)

In [0]:
# Calcula as métricas de avaliação no nível de produto
df_ranking_produtos_qualidade = (
    df_base_rankings_qualidade
    .select("id_pedido", "id_produto", "nome_produto", "categoria_produto", "nota_avaliacao")
    .dropDuplicates()
    .groupBy("id_produto", "nome_produto", "categoria_produto") # Agrupa os dados no nível do produto
    .agg(
        F.count("*").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media") 
    )
)

In [0]:
# Calcula as métricas de avaliação no nível de vendedor
df_ranking_vendedores_qualidade = (
    df_base_rankings_qualidade
    .select("id_pedido", "id_vendedor", "nome_vendedor", "estado", "nota_avaliacao")
    .dropDuplicates()
    .groupBy("id_vendedor", "nome_vendedor", "estado") # Agrupa os dados no nível do vendedor
    .agg(
        F.count("*").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media")
    )
)

##### Resultados:

In [0]:
# Seleciona o produto mais bem avaliado
df_produto_mais_bem_avaliado = (
    df_ranking_produtos_qualidade
    .orderBy(
        F.col("avaliacao_media").desc(),
        F.col("total_avaliacoes").desc()
    )
    .select("nome_produto", "categoria_produto", "avaliacao_media", "total_avaliacoes")
    .limit(1)
)

display(df_produto_mais_bem_avaliado)

nome_produto,categoria_produto,avaliacao_media,total_avaliacoes
Protetor Solar Branco,beleza_saude,5.0,13


In [0]:
# Seleciona o produto menos bem avaliado
df_produto_menos_bem_avaliado = (
    df_ranking_produtos_qualidade
    .orderBy(
        F.col("avaliacao_media").asc(),
        F.col("total_avaliacoes").desc()
    )
    .select("nome_produto", "categoria_produto", "avaliacao_media", "total_avaliacoes")
    .limit(1)
)

display(df_produto_menos_bem_avaliado)

nome_produto,categoria_produto,avaliacao_media,total_avaliacoes
Webcam Rosa,informatica_acessorios,1.0,8


In [0]:
# Seleciona o vendedor mais bem avaliado
df_vendedor_mais_bem_avaliado = (
    df_ranking_vendedores_qualidade
    .orderBy(
        F.col("avaliacao_media").desc(),
        F.col("total_avaliacoes").desc()
    )
    .select("nome_vendedor", "estado", "avaliacao_media", "total_avaliacoes")
    .limit(1)
)

display(df_vendedor_mais_bem_avaliado)

nome_vendedor,estado,avaliacao_media,total_avaliacoes
Luiz Otávio Abreu,PR,5.0,33


In [0]:
# Seleciona o vendedor menos bem avaliado
df_vendedor_menos_bem_avaliado = (
    df_ranking_vendedores_qualidade
    .orderBy(
        F.col("avaliacao_media").asc(),
        F.col("total_avaliacoes").desc()
    )
    .select("nome_vendedor", "estado", "avaliacao_media", "total_avaliacoes")
    .limit(1)
)

display(df_vendedor_menos_bem_avaliado)

nome_vendedor,estado,avaliacao_media,total_avaliacoes
Josué Ribeiro,ES,1.0,3
